In [2]:
import pandas as pd
import bw2data as bd
import bw2io as bi
import bw2calc as bc
import bw2analyzer as bwa
from itertools import product
import numpy as np
from tqdm import tqdm

16:29:52+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [3]:
bd.projects.set_current("2026-01-13-ecoinvent-3.11-cutoff")

In [4]:
bd.databases

Databases dictionary with 8 object(s):
	2026-3-19-11h18_processes_RE2020
	2026-4-21_consumptions_processes
	2026-4-22-15h1_buildings_RE2020
	2026-4-22-16h25_archetypes_RE2020_BB
	2026-4-22-16h7_buildings_cons_RE2020
	ecoinvent-3.11-biosphere
	ecoinvent-3.11-cutoff
	pr_based_arch_alamain

In [5]:
archdb = bd.Database("2026-4-22-16h25_archetypes_RE2020_BB")

In [6]:
keywords  = ["Maison",
"Logement",
"Bureau",
"primaire",
"secondaire"
]


In [8]:
procedes = [archdb.search(keyword)[0] for keyword in keywords]
procedes

['Maison_individuelle_ou_accolée' (m2, FR, None),
 'Logement_collectif' (m2, FR, None),
 'Bureau' (m2, FR, None),
 'Enseignement_primaire' (m2, FR, None),
 'Enseignement_secondaire_(partie_jour)' (m2, FR, None)]

In [11]:

[m for m in bd.methods if (("EF v3.1" in str(m)) 
    and ("no LT" not in str(m)))]

[('ecoinvent-3.11', 'EF v3.1', 'acidification', 'accumulated exceedance (AE)'),
 ('ecoinvent-3.11',
  'EF v3.1',
  'climate change',
  'global warming potential (GWP100)'),
 ('ecoinvent-3.11',
  'EF v3.1',
  'climate change: biogenic',
  'global warming potential (GWP100)'),
 ('ecoinvent-3.11',
  'EF v3.1',
  'climate change: fossil',
  'global warming potential (GWP100)'),
 ('ecoinvent-3.11',
  'EF v3.1',
  'climate change: land use and land use change',
  'global warming potential (GWP100)'),
 ('ecoinvent-3.11',
  'EF v3.1',
  'ecotoxicity: freshwater',
  'comparative toxic unit for ecosystems (CTUe)'),
 ('ecoinvent-3.11',
  'EF v3.1',
  'ecotoxicity: freshwater, inorganics',
  'comparative toxic unit for ecosystems (CTUe)'),
 ('ecoinvent-3.11',
  'EF v3.1',
  'ecotoxicity: freshwater, organics',
  'comparative toxic unit for ecosystems (CTUe)'),
 ('ecoinvent-3.11',
  'EF v3.1',
  'energy resources: non-renewable',
  'abiotic depletion potential (ADP): fossil fuels'),
 ('ecoinvent-3.

In [12]:
demands = {p["name"] : {p.id : 1.0} for p in procedes}
indicators = [m for m in bd.methods if (
    ("EF v3.1" in str(m)) 
    and ("no LT" not in str(m))
    and (
        (
        "gwp" in str(m).lower() 
        # and "land" in str(m).lower()
        and "EN15804+A2" not in str(m)
        )
        or ("eutrophication" in str(m).lower()) 
        or ("water use" in str(m).lower())
        or ("land use" in str(m).lower())
        )
    )
    ]

In [13]:
config = {"impact_categories" : indicators}
data_objs = bd.get_multilca_data_objs(functional_units=demands,method_config=config)
lca=bc.MultiLCA(demands=demands, method_config=config, data_objs=data_objs)
lca.lci()
lca.lcia()

Flux élémentaires qui contribuent de au moins à cutoff % d'un des scores d'impacts

In [14]:
cutoff = 0.01

def add_unique_elements(A, B):
    for element in B:
        if element not in A:
            A.append(element)
    return A

In [15]:
EF_to_keep = { p['name'] : [] for p in procedes}
keys = [(i,p["name"]) for i,p in product(indicators, procedes)]
for key in tqdm(keys) :
    s = lca.scores[key]
    contrib_EF = lca.characterized_inventories[key].sum(axis=1)
    nodes = [bd.get_node(id = i) for i in lca.dicts.biosphere.keys()]
    contrib_EF_dict = {nodes[i]:float(contrib_EF[i,0]) for i in range(len(nodes))}
    to_add = [k for k,v in contrib_EF_dict.items() if np.abs(v) > s*cutoff]
    # print(key,":",len(to_add))
    # print(to_add)
    add_unique_elements(EF_to_keep[key[1]],to_add)


  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:32<00:00,  1.52it/s]


Récupération des inventaires avec cutoff sur les scores d'impacts des méthodes sélectionnées

In [16]:
inventories = { p['name'] : [] for p in procedes}
keys = [p["name"] for p in procedes]
for key in tqdm(keys) :
    contrib_EF = lca.inventories[key].sum(axis=1)
    nodes = [bd.get_node(id = i) for i in lca.dicts.biosphere.keys()]
    contrib_EF_dict = {nodes[i]:float(contrib_EF[i,0]) for i in range(len(nodes))}
    inventories[key] = {k : v for k,v in contrib_EF_dict.items() if k in EF_to_keep[key]}


100%|██████████| 5/5 [00:03<00:00,  1.28it/s]


In [17]:
import pandas as pd
from datetime import datetime

today = datetime.today()
df = pd.DataFrame(inventories).to_excel(f"/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/data/processed/{today.year}-{today.month}-{today.day}_elem_flows_plus_contributeurs.xlsx")

Récupération des facteurs de caractérisation pour les méthodes sélectionnés et les flux élémentaires de l'inventaire restreint

In [18]:
efs = set([item for l in EF_to_keep.values() for item in l])

In [19]:
#Récupération des cfs 
cfs = {str(i) : {} for i in indicators}
keys = [(i,ef) for i,ef in product(indicators, efs)]
for i in indicators :
    char_mat = lca.characterization_matrices[i]
    vec_mat = char_mat.sum(axis=1)
    for ef in efs :
        row_index = lca.dicts.biosphere[ef.id]
        cfs[str(i)][str(ef)] = vec_mat[row_index].item()

    

In [20]:
pd.DataFrame(cfs).to_excel(f"/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/data/processed/{today.year}-{today.month}-{today.day}_cfs_plus_contributeurs.xlsx")

Analyse de contribution rapide

In [25]:
act = procedes[0]
act

'Maison_individuelle_ou_accolée' (m2, FR, None)

In [26]:
ind = indicators[-1]

In [27]:
indicators

[('ecoinvent-3.11',
  'EF v3.1',
  'climate change',
  'global warming potential (GWP100)'),
 ('ecoinvent-3.11',
  'EF v3.1',
  'climate change: biogenic',
  'global warming potential (GWP100)'),
 ('ecoinvent-3.11',
  'EF v3.1',
  'climate change: fossil',
  'global warming potential (GWP100)'),
 ('ecoinvent-3.11',
  'EF v3.1',
  'climate change: land use and land use change',
  'global warming potential (GWP100)'),
 ('ecoinvent-3.11',
  'EF v3.1',
  'eutrophication: freshwater',
  'fraction of nutrients reaching freshwater end compartment (P)'),
 ('ecoinvent-3.11',
  'EF v3.1',
  'eutrophication: marine',
  'fraction of nutrients reaching marine end compartment (N)'),
 ('ecoinvent-3.11',
  'EF v3.1',
  'eutrophication: terrestrial',
  'accumulated exceedance (AE)'),
 ('ecoinvent-3.11', 'EF v3.1', 'land use', 'soil quality index'),
 ('ecoinvent-3.11',
  'EF v3.1',
  'water use',
  'user deprivation potential (deprivation-weighted water consumption)'),
 ('ecoinvent-3.11',
  'EN15804+A2 

In [28]:
bwa.print_recursive_calculation(activity=act,lcia_method=ind)

Fraction of score | Absolute score | Amount | Activity
0001 | 3.329 |     1 | 'Maison_individuelle_ou_accolée' (m2, FR, None)


KeyboardInterrupt: 